In [ ]:
# ==============================================================================
# 1. 데이터 저장 폴더 생성
# ==============================================================================
!mkdir -p /content/DIV2K

# 1. Train HR 데이터 다운로드 및 압축 해제
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip -O /content/DIV2K/train_hr.zip
!unzip -q /content/DIV2K/train_hr.zip -d /content/DIV2K
!rm /content/DIV2K/train_hr.zip

# 2. Valid HR 데이터 다운로드 및 압축 해제
!wget http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip -O /content/DIV2K/valid_hr.zip
!unzip -q /content/DIV2K/valid_hr.zip -d /content/DIV2K
!rm /content/DIV2K/valid_hr.zip

print("데이터셋 다운로드 완료")

In [ ]:
# ==============================================================================
# 2. 라이브러리 임포트
# ==============================================================================
import os, glob, math, random
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ==============================================================================
# 3. 학습 환경 설정
# ==============================================================================
CONFIG = {
    "GPU_ID": 0,
    "DATA_PATH": "/content/DIV2K",

    # Data / training setting
    "HR_SIZE": 192,
    "SCALE_CHOICES_TRAIN": [2, 3, 4],
    "SCALE_VAL": 2,
    "BATCH_SIZE": 4,

    "EPOCHS": 400,
    "LEARNING_RATE": 4e-5,
    "USE_AMP": True,

    # Warmup
    "WARMUP_EPOCHS": 20,
    "WARMUP_MULTIPLIER": 3,

    # Coord sampling (your original intention)
    "TRAIN_COORD_SAMPLES": 4096,
    "VAL_EVERY": 1,
    "VIS_EVERY": 10,


    # Checkpoint cadence (important if session breaks)
    "CKPT_SAVE_EVERY_STEPS": 500,
    # AMP
    "USE_AMP": True,

    # data_norm: (x-0.5)/0.5
    "DATA_NORM": True,

    # experiment knobs
    "EXP_NAME": "SRNO_Base_final", # 폴더 명 설정
    "USE_FOURIER": False,          # Fourier 변환 실험
    "L": 5,                        # 주파수 설정
    "CORNER_AGG": "concat",        # "concat" or "inv_area"

    # model size knobs
    "WIDTH": 128,               # 96 or 64 to speed up
    "BLOCKS": 8,                # keep 8 for SRNO-like
    "RESIDUAL_SCALE": 0.1,
}

DEVICE = torch.device(f"cuda:{CONFIG['GPU_ID']}" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

SAVE_ROOT = '/content/drive/MyDrive'
SAVE_FOLDER = os.path.join(SAVE_ROOT, CONFIG["EXP_NAME"])
os.makedirs(SAVE_FOLDER, exist_ok=True)

IMG_SAVE_DIR = os.path.join(SAVE_FOLDER, "epoch_images")
os.makedirs(IMG_SAVE_DIR, exist_ok=True)

CKPT_PATH = os.path.join(SAVE_FOLDER, "latest_checkpoint.pth")
BEST_CKPT_PATH = os.path.join(SAVE_FOLDER, "best_checkpoint.pth")
BEST_MODEL_PATH = os.path.join(SAVE_FOLDER, "best_model.pth")

print(f"Results saved to: {SAVE_FOLDER}")

In [ ]:
# ==============================================================================
# 4. Utils
# ==============================================================================
def norm_01_to_m11(x):
    return (x - 0.5) / 0.5

def denorm_m11_to_01(x):
    return x * 0.5 + 0.5

def make_coord_grid(h, w, device, batch_size=1):
    y = torch.linspace(-1, 1, h, device=device)
    x = torch.linspace(-1, 1, w, device=device)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    coords = torch.stack([xx, yy], dim=-1).view(1, -1, 2)  # (1,N,2)
    cell = torch.tensor([2.0/h, 2.0/w], device=device, dtype=torch.float32).view(1, 2)
    if batch_size != 1:
        coords = coords.repeat(batch_size, 1, 1)
        cell = cell.repeat(batch_size, 1)
    return coords, cell

def sample_coord_grid(h, w, device, batch_size, n_samples):
    ys = torch.randint(0, h, (batch_size, n_samples), device=device)
    xs = torch.randint(0, w, (batch_size, n_samples), device=device)

    x_norm = (xs.float() / (w - 1)).mul(2).sub(1)
    y_norm = (ys.float() / (h - 1)).mul(2).sub(1)
    coords = torch.stack([x_norm, y_norm], dim=-1)  # (B,n_samples,2)

    cell = torch.tensor([2.0/h, 2.0/w], device=device, dtype=torch.float32).view(1,2).repeat(batch_size, 1)
    idx = ys * w + xs  # (B,n_samples)
    return coords, cell, idx

# ==============================================================================
# 5. PSNR (Y channel)
# ==============================================================================
def rgb_to_y(img):  # img (...,3) in [0,1]
    return 0.2567 * img[...,0] + 0.5041 * img[...,1] + 0.0979 * img[...,2] + 16/255

def calc_psnr_y_safe(pred, target, shave=2):
    pred = pred.clamp(0,1)
    target = target.clamp(0,1)

    if shave > 0:
        pred = pred[..., shave:-shave, shave:-shave]
        target = target[..., shave:-shave, shave:-shave]

    pred = pred.permute(0,2,3,1)
    target = target.permute(0,2,3,1)

    y_pred = rgb_to_y(pred)
    y_tgt  = rgb_to_y(target)

    diff = (y_pred - y_tgt)
    mse = torch.mean(diff * diff)

    if not torch.isfinite(mse):
        return float("nan")
    mse_val = float(mse.item())
    if mse_val <= 0:
        return 100.0
    return 20.0 * math.log10(1.0 / math.sqrt(mse_val))

In [ ]:
# ==============================================================================
# 6. Dataset: HR crop fixed, LR generated by scale (2/3/4)
# ==============================================================================
class DIV2KDataset(Dataset):
    def __init__(self, root_dir, hr_size=192, split="train"):
        self.split = split
        self.hr_size = hr_size

        if split == "train":
            pattern = os.path.join(root_dir, "DIV2K_train_HR", "*.png")
            self.files = sorted(glob.glob(pattern))
            self.transform = transforms.Compose([
                transforms.RandomCrop(hr_size),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.ToTensor(),
            ])
            print(f"[Train] {len(self.files)} images")
        else:
            pattern = os.path.join(root_dir, "DIV2K_valid_HR", "*.png")
            self.files = sorted(glob.glob(pattern))
            self.transform = transforms.Compose([
                transforms.CenterCrop(hr_size),
                transforms.ToTensor(),
            ])
            print(f"[Valid] {len(self.files)} images")

        assert len(self.files) > 0, "DIV2K 파일 경로 확인!"

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        hr = self.transform(img)  # (3,HR,HR) in [0,1]
        if CONFIG["DATA_NORM"]:
            hr = norm_01_to_m11(hr)
        return hr

# ==============================================================================
# 7. Warmup Scheduler
# ==============================================================================
class WarmupScheduler:
    def __init__(self, optimizer, base_lr, multiplier=5, warmup_epochs=20):
        self.optimizer = optimizer
        self.base_lr = base_lr
        self.multiplier = multiplier
        self.warmup_epochs = warmup_epochs
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self._apply_lr()

    def _apply_lr(self):
        if self.epoch <= 0:
            factor = 1.0
        elif self.epoch <= self.warmup_epochs:
            factor = 1 + (self.multiplier - 1) * (self.epoch / self.warmup_epochs)
        else:
            factor = self.multiplier
        for g in self.optimizer.param_groups:
            g["lr"] = self.base_lr * factor

    def step(self):
        self.epoch += 1
        self._apply_lr()

    def get_lr(self):
        return self.optimizer.param_groups[0]["lr"]

In [ ]:
# ==============================================================================
# 8. Model (SRNO-like): EDSR(no upsampling) + Galerkin + coord/cell
#    - Fourier positional encoding (optional)
#    - corner aggregation: concat / inv_area
# ==============================================================================
class FourierPositionalEncoding(nn.Module):
    def __init__(self, L=5):
        super().__init__()
        self.L = L
        self.register_buffer("freq_bands", 2 ** torch.linspace(0, L-1, L), persistent=False)

    def forward(self, x):  # (B,N,2)
        pe = []
        for freq in self.freq_bands.to(x.device):
            pe.append(torch.sin(x * freq * math.pi))
            pe.append(torch.cos(x * freq * math.pi))
        return torch.cat(pe, dim=-1)  # (B,N, 2*L*2)

class GalerkinAttention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        assert dim % heads == 0
        self.heads = heads
        self.head_dim = dim // heads

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        # Eq.12 style: LN on K,V
        self.ln_k = nn.LayerNorm(dim)
        self.ln_v = nn.LayerNorm(dim)

        self.to_out = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape

        q = self.to_q(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)  # (B,H,N,D)
        k = self.to_k(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.to_v(x).reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)

        # LN at token level
        k2 = k.permute(0, 2, 1, 3).reshape(B, N, C)
        v2 = v.permute(0, 2, 1, 3).reshape(B, N, C)
        k2 = self.ln_k(k2)
        v2 = self.ln_v(v2)
        k = k2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)
        v = v2.reshape(B, N, self.heads, self.head_dim).permute(0, 2, 1, 3)

        # Galerkin: Q (K^T V) / N
        context = torch.matmul(k.transpose(-2, -1), v) / float(N)  # (B,H,D,D)
        out = torch.matmul(q, context)                             # (B,H,N,D)

        out = out.permute(0, 2, 1, 3).reshape(B, N, C)
        return self.to_out(out)

class ResBlock(nn.Module):
    def __init__(self, n_feats, kernel_size=3, act=nn.ReLU(True), res_scale=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
            act,
            nn.Conv2d(n_feats, n_feats, kernel_size, padding=kernel_size//2),
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.body(x) * self.res_scale

class EDSR(nn.Module):
    def __init__(self, n_resblocks=16, n_feats=64):
        super().__init__()
        self.head = nn.Sequential(nn.Conv2d(3, n_feats, 3, padding=1))
        m_body = [ResBlock(n_feats, 3, res_scale=0.1) for _ in range(n_resblocks)]
        m_body.append(nn.Conv2d(n_feats, n_feats, 3, padding=1))
        self.body = nn.Sequential(*m_body)

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        return x + res

class SRNO_Fourier(nn.Module):
    def __init__(self, residual_scale=0.1, use_fourier=False, L=5, width=128, blocks=8, corner_agg="concat"):
        super().__init__()
        assert corner_agg in ["concat", "inv_area"], "corner_agg must be 'concat' or 'inv_area'"
        self.corner_agg = corner_agg

        self.encoder = EDSR(16, 64)
        self.use_fourier = use_fourier
        self.pos_enc = FourierPositionalEncoding(L=L)

        # corner feature dim = q_feat(64) + rel_coord(2) + rel_cell(2) + (optional) fourier(4L)
        corner_dim = 64 + 2 + 2
        if use_fourier:
            corner_dim += (2 * L * 2)

        in_dim = (4 * corner_dim) if (self.corner_agg == "concat") else corner_dim

        self.latent_dim = width
        self.lifting = nn.Linear(in_dim, self.latent_dim)

        self.layers = nn.ModuleList([GalerkinAttention(self.latent_dim, heads=8) for _ in range(blocks)])
        self.ffns = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(self.latent_dim),
                nn.Linear(self.latent_dim, self.latent_dim),
                nn.GELU(),
                nn.Linear(self.latent_dim, self.latent_dim),
            ) for _ in range(blocks)
        ])

        self.projection = nn.Linear(self.latent_dim, 3)
        self.residual_scale = residual_scale

    @staticmethod
    def _feat_coord_grid(Hf, Wf, device, B):
        y = torch.linspace(-1, 1, Hf, device=device, dtype=torch.float32)
        x = torch.linspace(-1, 1, Wf, device=device, dtype=torch.float32)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        grid = torch.stack([xx, yy], dim=-1).unsqueeze(0)  # (1,Hf,Wf,2)
        return grid.repeat(B, 1, 1, 1)  # (B,Hf,Wf,2)

    def forward(self, x, coords, cell):
        """
        x: (B,3,Hlr,Wlr)  (DATA_NORM이면 [-1,1])
        coords: (B,N,2)   HR grid coords in [-1,1]
        cell: (B,2)       [2/Hhr, 2/Whr] * scale  (SRNO-style)
        return: (B,N,3)   (same norm space)
        """
        B = x.shape[0]

        coords = coords.to(dtype=torch.float32)
        cell   = cell.to(dtype=torch.float32)

        grid = coords.unsqueeze(2)  # (B,N,1,2)

        # base from LR (bilinear)
        base = F.grid_sample(x, grid, mode="bilinear", align_corners=False)  # (B,3,N,1)
        base = base.squeeze(3).permute(0,2,1)  # (B,N,3)

        feat = self.encoder(x)  # (B,64,Hf,Wf)
        _, _, Hf, Wf = feat.shape

        feat_coord = self._feat_coord_grid(Hf, Wf, feat.device, B).permute(0,3,1,2)  # (B,2,Hf,Wf)

        rx, ry = 1.0 / Hf, 1.0 / Wf
        eps = 1e-6
        vx_lst = [-1, 1]
        vy_lst = [-1, 1]

        # cell -> feature-grid scale (SRNO-style)
        rel_cell = cell.unsqueeze(1).repeat(1, coords.shape[1], 1).clone()
        rel_cell[..., 0] *= Hf
        rel_cell[..., 1] *= Wf

        preds = []
        areas = []
        for vx in vx_lst:
            for vy in vy_lst:
                coord_ = coords.clone()
                coord_[..., 0] = (coord_[..., 0] + vx * rx).clamp(-1 + eps, 1 - eps)
                coord_[..., 1] = (coord_[..., 1] + vy * ry).clamp(-1 + eps, 1 - eps)
                grid_ = coord_.unsqueeze(2)

                q_feat = F.grid_sample(feat, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)
                q_coord = F.grid_sample(feat_coord, grid_, mode="nearest", align_corners=False).squeeze(3).permute(0,2,1)

                rel_coord = coords - q_coord
                rel_coord[..., 0] *= Hf
                rel_coord[..., 1] *= Wf

                # inv_area proxy
                area = (rel_coord[..., 0].abs() * rel_coord[..., 1].abs())
                areas.append(area)

                parts = [q_feat, rel_coord]
                if self.use_fourier:
                    parts.append(self.pos_enc(rel_coord))
                parts.append(rel_cell)

                preds.append(torch.cat(parts, dim=-1))  # (B,N,corner_dim)

        if self.corner_agg == "concat":
            z_in = torch.cat(preds, dim=-1)  # (B,N,4*corner_dim)
        else:
            eps_w = 1e-9
            inv = [1.0 / (a + eps_w) for a in areas]
            inv_sum = (inv[0] + inv[1] + inv[2] + inv[3]).clamp_min(eps_w)
            w = [(iv / inv_sum).unsqueeze(-1) for iv in inv]
            z_in = preds[0]*w[0] + preds[1]*w[1] + preds[2]*w[2] + preds[3]*w[3]  # (B,N,corner_dim)

        z = self.lifting(z_in)

        for attn, ffn in zip(self.layers, self.ffns):
            z = z + attn(z)
            z = z + ffn(z)

        residual = self.projection(z) * self.residual_scale
        return base + residual

In [ ]:
# ==============================================================================
# 9. Train/Val loop (checkpoint + visualization)
# ==============================================================================
def train():
    train_dataset = DIV2KDataset(CONFIG["DATA_PATH"], hr_size=CONFIG["HR_SIZE"], split="train")
    valid_dataset = DIV2KDataset(CONFIG["DATA_PATH"], hr_size=CONFIG["HR_SIZE"], split="valid")

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["BATCH_SIZE"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )
    val_loader = DataLoader(
        valid_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )

    # fixed sample for visualization (HR only)
    fixed_hr = valid_dataset[0]  # (3,HR,HR) already normalized if DATA_NORM
    fixed_hr = fixed_hr.unsqueeze(0).to(DEVICE, non_blocking=True)  # (1,3,HR,HR)

    fixed_scale = float(CONFIG["SCALE_VAL"])  # 2.0 (val 고정)
    lr_size = int(round(CONFIG["HR_SIZE"] / fixed_scale))

    # make LR from fixed HR (same way as training/validation)
    fixed_lr = F.interpolate(
        fixed_hr, size=(lr_size, lr_size),
        mode="bicubic", align_corners=False
    )
    fixed_scale_t = torch.tensor([fixed_scale], device=DEVICE, dtype=torch.float32)  # (1,)

    model = SRNO_Fourier(
        residual_scale=CONFIG["RESIDUAL_SCALE"],
        use_fourier=CONFIG["USE_FOURIER"],
        L=CONFIG["L"],
        width=CONFIG["WIDTH"],
        blocks=CONFIG["BLOCKS"],
        corner_agg=CONFIG["CORNER_AGG"],
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LEARNING_RATE"])
    scheduler = WarmupScheduler(
        optimizer,
        base_lr=CONFIG["LEARNING_RATE"],
        multiplier=CONFIG["WARMUP_MULTIPLIER"],
        warmup_epochs=CONFIG["WARMUP_EPOCHS"],
    )

    criterion = nn.L1Loss()
    scaler = torch.amp.GradScaler("cuda", enabled=(CONFIG["USE_AMP"] and torch.cuda.is_available()))

    start_epoch = 0
    best_psnr = -1.0

    # resume
    if os.path.exists(CKPT_PATH):
        print("🔁 Resume from checkpoint:", CKPT_PATH)
        ckpt = torch.load(CKPT_PATH, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = int(ckpt.get("epoch", 0))
        best_psnr = float(ckpt.get("best_psnr", best_psnr))
        scheduler.set_epoch(int(ckpt.get("scheduler_epoch", start_epoch)))

        if "scaler_state_dict" in ckpt and ckpt["scaler_state_dict"] is not None:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        print(f"   → start_epoch={start_epoch} | best_psnr={best_psnr:.2f} | lr={scheduler.get_lr():.2e}")

    print("🚀 Training Start")
    print(f"🧩 Train coord samples: {CONFIG['TRAIN_COORD_SAMPLES']}")
    print(f"📌 Train scale choices: {CONFIG['SCALE_CHOICES_TRAIN']} | Val scale={CONFIG['SCALE_VAL']}")
    print(f"🧪 EXP: {CONFIG['EXP_NAME']} | Fourier={CONFIG['USE_FOURIER']}(L={CONFIG['L']}) | corner_agg={CONFIG['CORNER_AGG']}")
    print(f"🧱 Model: width={CONFIG['WIDTH']} blocks={CONFIG['BLOCKS']}")

    for epoch in range(start_epoch, CONFIG["EPOCHS"]):
        # ---------------- Train ----------------
        model.train()
        loss_sum = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}", leave=False)

        for hr in pbar:
            hr = hr.to(DEVICE, non_blocking=True)  # (B,3,192,192)

            # ✅ 배치 단위 scale 선택
            scale = float(random.choice(CONFIG["SCALE_CHOICES_TRAIN"]))
            lr_size = int(round(CONFIG["HR_SIZE"] / scale))

            # ✅ LR 생성 (배치 내 모두 동일 크기)
            lr = F.interpolate(hr, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

            B = lr.shape[0]
            H, W = hr.shape[2], hr.shape[3]

            # ✅ HR 기준 coord sampling
            coords, cell, idx = sample_coord_grid(H, W, DEVICE, batch_size=B, n_samples=CONFIG["TRAIN_COORD_SAMPLES"])

            scale_t = torch.full((B, 1), scale, device=DEVICE, dtype=torch.float32)
            cell = cell * scale_t

            # GT gather at sampled coords
            target_full = hr.permute(0,2,3,1).reshape(B, -1, 3)
            gather_idx = idx.unsqueeze(-1).expand(-1, -1, 3)
            target = torch.gather(target_full, dim=1, index=gather_idx)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(CONFIG["USE_AMP"] and torch.cuda.is_available())):
                pred = model(lr, coords, cell)      # (B, Ns, 3)
                loss = criterion(pred, target)

            if not torch.isfinite(loss):
                print(f"⚠️ non-finite loss detected at epoch {epoch+1}, skipping batch. loss={loss}")
                optimizer.zero_grad(set_to_none=True)
                scaler.update()
                continue

            # backward
            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            loss_sum += float(loss.item())
            pbar.set_postfix(loss=float(loss.item()), lr=scheduler.get_lr())

        scheduler.step()
        avg_loss = loss_sum / max(len(train_loader), 1)

        # ---------------- Val ----------------
        do_val = ((epoch + 1) % CONFIG["VAL_EVERY"] == 0) or ((epoch + 1) == 1)
        if do_val:
            model.eval()
            psnr_sum, bic_psnr_sum, cnt = 0.0, 0.0, 0

            with torch.no_grad():
                for hr in val_loader:
                    hr = hr.to(DEVICE, non_blocking=True)  # (1,3,192,192)

                    scale = float(CONFIG["SCALE_VAL"])     # 2.0
                    lr_size = int(round(CONFIG["HR_SIZE"] / scale))
                    lr = F.interpolate(hr, size=(lr_size, lr_size), mode="bicubic", align_corners=False)

                    coords, cell = make_coord_grid(hr.shape[2], hr.shape[3], DEVICE, batch_size=1)
                    scale_t = torch.tensor([[scale]], device=DEVICE, dtype=torch.float32)  # (1,1)
                    cell = cell * scale_t

                    pred = model(lr, coords, cell)  # (1,N,3)

                    pred_img = pred.reshape(hr.shape[2], hr.shape[3], 3).permute(2,0,1).unsqueeze(0)

                    if CONFIG["DATA_NORM"]:
                        pred_img_01 = denorm_m11_to_01(pred_img)
                        hr_01 = denorm_m11_to_01(hr)
                        lr_01 = denorm_m11_to_01(lr)
                    else:
                        pred_img_01 = pred_img
                        hr_01 = hr
                        lr_01 = lr

                    bic = F.interpolate(lr_01, size=hr_01.shape[-2:], mode="bicubic", align_corners=False)

                    psnr = calc_psnr_y_safe(pred_img_01, hr_01, shave=2)
                    bic_psnr = calc_psnr_y_safe(bic, hr_01, shave=2)

                    if not (math.isnan(psnr) or math.isnan(bic_psnr)):
                        psnr_sum += psnr
                        bic_psnr_sum += bic_psnr
                        cnt += 1

            avg_psnr = psnr_sum / max(cnt, 1)
            avg_bic_psnr = bic_psnr_sum / max(cnt, 1)

            print(f"[Epoch {epoch+1:03d}] loss={avg_loss:.5f} | Val Y-PSNR={avg_psnr:.2f} | Bicubic={avg_bic_psnr:.2f}")

            # -------- Visualization (fixed sample) --------
            vis_every = int(CONFIG.get("VIS_EVERY", 10))  # 없으면 10 Epochs마다
            do_vis = ((epoch + 1) % vis_every == 0) or ((epoch + 1) == 1)

            if do_vis:
                with torch.no_grad():
                    coords_f, cell_f = make_coord_grid(fixed_hr.shape[2], fixed_hr.shape[3], DEVICE, batch_size=1)
                    cell_f = cell_f * fixed_scale_t.view(1, 1)

                    pred_f = model(fixed_lr, coords_f, cell_f)  # (1,N,3)
                    pred_img_f = pred_f.reshape(fixed_hr.shape[2], fixed_hr.shape[3], 3).permute(2,0,1).unsqueeze(0)

                    if CONFIG["DATA_NORM"]:
                        pred_vis = denorm_m11_to_01(pred_img_f)
                        hr_vis = denorm_m11_to_01(fixed_hr)
                        lr_vis = denorm_m11_to_01(fixed_lr)
                    else:
                        pred_vis, hr_vis, lr_vis = pred_img_f, fixed_hr, fixed_lr

                    lr_np = lr_vis[0].permute(1,2,0).detach().cpu().numpy()
                    sr_np = pred_vis[0].permute(1,2,0).detach().cpu().numpy()
                    gt_np = hr_vis[0].permute(1,2,0).detach().cpu().numpy()

                    lr_np = np.nan_to_num(lr_np, nan=0.0, posinf=1.0, neginf=0.0)
                    sr_np = np.nan_to_num(sr_np, nan=0.0, posinf=1.0, neginf=0.0)
                    gt_np = np.nan_to_num(gt_np, nan=0.0, posinf=1.0, neginf=0.0)

                    fig = plt.figure(figsize=(15,5))
                    fig.suptitle(
                        f"{CONFIG['EXP_NAME']} | Epoch {epoch+1:03d} | Val Y-PSNR {avg_psnr:.2f} dB",
                        fontsize=14
                    )

                    ax1 = fig.add_subplot(1,3,1); ax1.imshow(np.clip(lr_np,0,1)); ax1.set_title("LR"); ax1.axis("off")
                    ax2 = fig.add_subplot(1,3,2); ax2.imshow(np.clip(sr_np,0,1)); ax2.set_title("SR"); ax2.axis("off")
                    ax3 = fig.add_subplot(1,3,3); ax3.imshow(np.clip(gt_np,0,1)); ax3.set_title("GT"); ax3.axis("off")

                    save_path = os.path.join(IMG_SAVE_DIR, f"epoch_{epoch+1:03d}_psnr_{avg_psnr:.2f}.png")
                    fig.savefig(save_path, bbox_inches="tight")
                    plt.show()
                    plt.close(fig)

            # -------- Save BEST --------
            if avg_psnr > best_psnr:
                best_psnr = avg_psnr
                torch.save(model.state_dict(), BEST_MODEL_PATH)
                torch.save({
                    "epoch": epoch + 1,
                    "best_psnr": best_psnr,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_epoch": scheduler.epoch,
                    "scaler_state_dict": scaler.state_dict(),
                    "config": CONFIG,
                }, BEST_CKPT_PATH)
                print(f"🏆 NEW BEST: {best_psnr:.2f} dB (saved)")

        # -------- Save LAST ckpt (always) --------
        torch.save({
            "epoch": epoch + 1,
            "best_psnr": best_psnr,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_epoch": scheduler.epoch,
            "scaler_state_dict": scaler.state_dict(),
            "config": CONFIG,
        }, CKPT_PATH)

    print("✅ Training Done")

In [ ]:
train()